# Demo 6: Great Expectations

**Kapcsolódó diák:** 46–49 (GE architektúra, Expectation Suite, Checkpoint, Data Docs)

**Előfeltétel:** `docker compose up -d`  
JupyterLab: http://localhost:8888 (token: `demo`)  
PostgreSQL: `localhost:5432` (user: labor, pass: labor, db: week06)

**Fontos:** a cellákat sorban kell futtatni – az állapot a `gx/` mappában perzisztál.

Tartalom:
1. Data Context inicializálás
2. Expectation Suite definiálása
3. Checkpoint futtatás – hibás adaton
4. Data Docs generálása
5. Airflow integráció kódpélda


In [1]:
!pip install -q great_expectations pandas pyarrow sqlalchemy psycopg2-binary

## 1. Data Context inicializálás

A `gx/` mappa tartalmazza a GE projekt konfigurációját.
A `mode='file'` azt jelenti, hogy a konfig fájlban perzisztál (nem csak memóriában).


In [2]:
import great_expectations as gx
import pandas as pd
import shutil, os

# Tiszta állapotból indul – korábbi gx/ mappa törlése
if os.path.exists('gx'):
    shutil.rmtree('gx')

# GE 1.x API: context.data_sources (nem context.sources)
context = gx.get_context(mode='file')
datasource = context.data_sources.add_pandas(name='orders_source')
data_asset = datasource.add_dataframe_asset(name='orders_df')
batch_definition = data_asset.add_batch_definition_whole_dataframe('orders_batch')

print('✅ Data Context inicializálva')
print(f'   Projekt mappa: {context.root_directory}')

✅ Data Context inicializálva
   Projekt mappa: /home/jovyan/work/gx


## 2. Expectation Suite definiálása

Az elvárások deklaratív módon íródnak – verziókontrollálhatók és dokumentáltak.
GE 1.x-ben az elvárásokat közvetlenül a Suite objektumhoz adjuk, validator nélkül.
A `mostly` paraméterrel részleges megfelelés is elfogadható.


In [3]:
SUITE_NAME = 'orders_quality_suite'

df_valid = pd.DataFrame({
    'order_id':    [1, 2, 3, 4, 5],
    'customer_id': [101, 102, 103, 104, 105],
    'amount':      [12500.0, 3200.0, 8750.0, 450.0, 15000.0],
    'status':      ['shipped', 'pending', 'shipped', 'cancelled', 'shipped'],
    'email':       ['a@b.hu', 'c@d.hu', 'e@f.hu', 'g@h.hu', 'i@j.hu'],
    'order_date':  pd.to_datetime(['2024-01-15','2024-01-16','2024-01-17','2024-01-18','2024-01-19']),
})

# GE 1.x: suite létrehozása és elvárások hozzáadása
suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME))

# Tábla szintű
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=10_000_000))
suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchOrderedList(
        column_list=['order_id', 'customer_id', 'amount', 'status', 'email', 'order_date']))

# Oszlop szintű – completeness
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='order_id'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='customer_id'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='amount'))

# Uniqueness
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column='order_id'))

# Validity
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='amount', min_value=0.01, max_value=10_000_000))
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='status', value_set=['shipped', 'pending', 'cancelled', 'returned']))
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='email',
        regex=r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$',
        mostly=0.95))
suite.add_expectation(
    gx.expectations.ExpectColumnMeanToBeBetween(
        column='amount', min_value=100.0, max_value=100_000.0))

# Validation Definition + gyors teszt valid adaton
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='orders_validation',
        data=batch_definition,
        suite=suite,
    )
)

quick = validation_def.run(batch_parameters={'dataframe': df_valid})
print(f'✅ Suite mentve: {SUITE_NAME}')
print(f'   Elvárások száma: {len(suite.expectations)}')
print(f'   Valid adaton: {"✅ PASSED" if quick.success else "❌ FAILED"}')

Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]

✅ Suite mentve: orders_quality_suite
   Elvárások száma: 10
   Valid adaton: ✅ PASSED


## 3. Checkpoint – hibás adaton

A hibás adaton futtatjuk a checkpointot: duplikált `order_id`, NULL `customer_id`,
negatív `amount`, érvénytelen `status` és `email`.


In [4]:
CHECKPOINT_NAME = 'orders_checkpoint'

df_bad = pd.DataFrame({
    'order_id':    [1, 2, 2, 4, 5],
    'customer_id': [101, None, 103, 104, 105],
    'amount':      [12500.0, 3200.0, -100.0, 450.0, 15000.0],
    'status':      ['shipped', 'SHIPPED', 'pending', 'cancelled', 'unknown'],
    'email':       ['a@b.hu', 'c@d.hu', 'nem_email', 'g@h.hu', 'i@j.hu'],
    'order_date':  pd.to_datetime(['2024-01-15','2024-01-16','2024-01-17','2024-01-18','2024-01-19']),
})

# GE 1.x: Checkpoint létrehozása ValidationDefinition-ből
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name=CHECKPOINT_NAME,
        validation_definitions=[validation_def],
    )
)

result = checkpoint.run(batch_parameters={'dataframe': df_bad})

# Eredmény kiértékelése
val_results  = list(result.run_results.values())
expectations = val_results[0].results if val_results else []

passed = sum(1 for r in expectations if r.success)
total  = len(expectations)

print(f"\n{'='*50}")
print(f"Checkpoint: {'✅ PASSED' if result.success else '❌ FAILED'}")
print(f"Elvárások: {passed}/{total} teljesült")
print(f"{'='*50}")

for r in expectations:
    icon = '✅' if r.success else '❌'
    cfg  = r.expectation_config
    # GE 1.x: .type attribútum (régebbi: .expectation_type)
    exp_type = getattr(cfg, 'type', None) or getattr(cfg, 'expectation_type', 'unknown')
    col      = (cfg.kwargs or {}).get('column', 'TABLE')
    print(f'  {icon} [{col:15}] {exp_type}')

Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]


Checkpoint: ❌ FAILED
Elvárások: 5/10 teljesült
  ✅ [TABLE          ] expect_table_row_count_to_be_between
  ✅ [TABLE          ] expect_table_columns_to_match_ordered_list
  ✅ [order_id       ] expect_column_values_to_not_be_null
  ❌ [order_id       ] expect_column_values_to_be_unique
  ❌ [customer_id    ] expect_column_values_to_not_be_null
  ✅ [amount         ] expect_column_values_to_not_be_null
  ❌ [amount         ] expect_column_values_to_be_between
  ✅ [amount         ] expect_column_mean_to_be_between
  ❌ [status         ] expect_column_values_to_be_in_set
  ❌ [email          ] expect_column_values_to_match_regex


## 4. Data Docs generálása

A Data Docs statikus HTML riport – S3-ra feltölthető, GitHub Pages-en publikálható.


In [5]:
context.build_data_docs()

docs_sites = context.get_docs_sites_urls()
for site in docs_sites:
    print(f'Data Docs: {site["site_url"]}')

# Böngészőben megnyitás (helyi futtatásnál):
context.open_data_docs()

Data Docs: file:///home/jovyan/work/gx/uncommitted/data_docs/local_site/index.html


## 5. Airflow integráció – kódpélda

A következő kód bemutatja, hogyan épül be a GE Checkpoint egy Airflow DAG-ba.
**Ez csak kódpélda – nem futtatható közvetlenül a notebookban.**


In [6]:
# ── Airflow DAG kódpélda – NEM FUTTATHATÓ ────────────────────────────────
# Csak olvasásra! Az Airflow környezetben (docker compose Airflow-zal) működne.

AIRFLOW_DAG_CODE = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import great_expectations as gx
import pandas as pd
import sys

def run_dq_checkpoint(**context):
    # GE 1.x API
    gx_context = gx.get_context(mode="file", project_root_dir="/opt/airflow/gx")
    df = pd.read_parquet("/opt/airflow/data/orders_bronze.parquet")
    checkpoint = gx_context.checkpoints.get("orders_checkpoint")
    result = checkpoint.run(batch_parameters={"dataframe": df})
    if not result.success:
        print("DQ FAILED – pipeline megallitva")
        sys.exit(1)   # Airflow task FAILED, downstream blokkolva
    print("DQ PASSED")

with DAG(
    dag_id="dq_orders_pipeline",
    default_args={"owner": "data-engineering", "retries": 0},
    schedule_interval="0 2 * * *",
    start_date=datetime(2024, 1, 1),
    catchup=False,
) as dag:
    dq_gate = PythonOperator(task_id="dq_bronze_gate", python_callable=run_dq_checkpoint)
'''

print('Airflow DAG kodpelda (reszlet):')
print(AIRFLOW_DAG_CODE[:600] + '...')

Airflow DAG kodpelda (reszlet):

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import great_expectations as gx
import pandas as pd
import sys

def run_dq_checkpoint(**context):
    # GE 1.x API
    gx_context = gx.get_context(mode="file", project_root_dir="/opt/airflow/gx")
    df = pd.read_parquet("/opt/airflow/data/orders_bronze.parquet")
    checkpoint = gx_context.checkpoints.get("orders_checkpoint")
    result = checkpoint.run(batch_parameters={"dataframe": df})
    if not result.success:
        print("DQ FAILED – pipeline megallitva")
        sys.exit(1)   # ...
